In [1]:
import music_library_tools.library as library
from music_library_tools.matching import find_candidate_originals
import pandas as pd

In [2]:
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)

In [3]:
xml_path = "/Users/stephenusher/Documents/Steve's Folder/music-library-tools/exports/raw/"

In [4]:
xml_file_current = xml_path + "MusicLibrary_2026_08_11.xml"
playlists_xml = library.get_library_section(library.load_library(xml_file_current), 'Playlists')
playlists = library.parse_array_elements(playlists_xml)
current_df = library.library_export_to_dataframe(xml_file_current)

In [5]:
xml_file_baseline = xml_path + "MusicLibrary_2026_08_03.xml"
baseline_df = library.library_export_to_dataframe(xml_file_baseline)

In [6]:
original_pool = baseline_df[baseline_df['Track ID'] == baseline_df['Min Track ID']]

In [7]:
playlist_stats = []
for code, name in library.CLOUD_STATUS_PLAYLISTS.items():
    
    playlist = library.get_playlist(playlists, name)
    if 'Playlist Items' in playlist:
        tracks = playlist['Playlist Items']
        track_count = len(tracks)
        duplicates = 0
        for track in tracks:
            track = library.get_track_by_id(current_df, track['Track ID'])
            has_duplicate = None
            if track is not None:
                has_duplicate = find_candidate_originals(track, original_pool)
                if not has_duplicate.empty:
                    duplicates += 1
        playlist_info = {
            'Name': code,
            'Track Count': track_count,
            'Candidate Track Count': duplicates,
            'Tracks Without Candidates': track_count - duplicates
        }
        playlist_stats.append(playlist_info)
    
cloud_playlist_stats = pd.DataFrame(playlist_stats)
print(cloud_playlist_stats.to_markdown(index=False))

| Name        |   Track Count |   Candidate Track Count |   Tracks Without Candidates |
|:------------|--------------:|------------------------:|----------------------------:|
| apple_music |          1845 |                     431 |                        1414 |
| error       |            32 |                       0 |                          32 |
| ineligible  |             8 |                       0 |                           8 |
| matched     |          5103 |                    5014 |                          89 |
| purchased   |             0 |                       0 |                           0 |
| removed     |          8449 |                       0 |                        8449 |
| uploaded    |            31 |                       1 |                          30 |
| waiting     |          3459 |                    3421 |                          38 |


In [8]:
playlist = library.get_playlist(playlists, library.CLOUD_STATUS_PLAYLISTS['removed'])

In [9]:
tracks = playlist['Playlist Items']

In [10]:
first_track = library.get_track_by_id(current_df, 15085)
print(first_track.to_markdown())

|                       | 6593                                                                                                                   |
|:----------------------|:-----------------------------------------------------------------------------------------------------------------------|
| Track ID              | 15085                                                                                                                  |
| Name                  | She's Only Happy When She's Dancin'                                                                                    |
| Artist                | Bryan Adams                                                                                                            |
| Album                 | Reckless                                                                                                               |
| Genre                 | Rock                                                                                        

In [14]:
cloud_playlist_stats.sum()

Name                         apple_musicerrorineligiblematchedpurchasedremoveduploadedwaiting
Track Count                                                                             18927
Candidate Track Count                                                                    8867
Tracks Without Candidates                                                               10060
dtype: object

In [15]:
len(current_df)

19028

In [16]:
19028 - 18927

101

In [17]:
playlist_track_ids = set()

for playlist_name in library.CLOUD_STATUS_PLAYLISTS.values():
    playlist = library.get_playlist(playlists, playlist_name)
    playlist_track_ids.update(library.get_playlist_track_ids(playlist))

In [18]:
current_track_ids = set(current_df['Track ID'])

In [19]:
missing_track_ids = current_track_ids - playlist_track_ids

In [21]:
missing_tracks = current_df[current_df['Track ID'].isin(missing_track_ids)]

In [29]:
missing_tracks.iloc[0]

Track ID                                                             23037
Name                                                     Love Of Your Life
Artist                                                                RAYE
Album                                           Love Of Your Life - Single
Genre                                                                  Pop
Kind                                            Apple Music AAC audio file
Size                                                               7174864
Total Time                                                          196171
Disc Number                                                            1.0
Disc Count                                                             1.0
Track Number                                                           1.0
Track Count                                                            1.0
Year                                                                2020.0
BPM                      

In [30]:
missing_tracks['Playlist Only'].value_counts()

Playlist Only
True    101
Name: count, dtype: int64

In [31]:
current_df['Playlist Only'].value_counts()

Playlist Only
True    101
Name: count, dtype: int64